# Check Grokking

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import sys

sys.path.append("..")
from src.utils import load_json

data1 = load_json(f"../result/grokking/weight_decay/grokking_00.json")
data2 = load_json(f"../result/grokking/weight_decay/grokking_25.json")
data3 = load_json(f"../result/grokking/weight_decay/grokking_50.json")
data4 = load_json(f"../result/grokking/weight_decay/grokking_75.json")

bunch = [data1, data2, data3, data4]

In [ ]:
def smoothing(data, window_len=20):
    window = np.ones(window_len) / window_len
    return np.convolve(data, window, mode='valid')

In [ ]:
# Train and Test Accuracy
train_accuracy, test_accuracy, generalization_error = [], [], []
for data in bunch:
    tmp1 = np.array(data['train_accuracy'])
    tmp2 = np.array(data['test_accuracy'])
    train_accuracy.append(tmp1)
    test_accuracy.append(tmp2)
    generalization_error.append(tmp1 - tmp2)

# Train and Test Target Linearity
train_target_linearity, test_target_linearity, tl_gap = [], [], []
for data in bunch:
    tmp = np.clip(np.array(data['train_target_linearity']).T, min=0.0)
    train_target_linearity.append(tmp)
    tmp1 = tmp[-1] - tmp[-2]
    tl_gap.append(tmp1)
    tmp = np.clip(np.array(data['test_target_linearity']).T, min=0.0)
    test_target_linearity.append(tmp)

In [ ]:
n_layers = train_target_linearity[0].shape[0]
fig, axes = plt.subplots(ncols=2, nrows=2, figsize=(6.5, 6.5), sharey=True)
cmap = cm.viridis
colors = [cmap(1 - (i / (n_layers - 1))) for i in range(n_layers)]

titles = ['$\\lambda=0.0$', '$\\lambda=0.25$', '$\\lambda=0.5$', '$\\lambda=0.75$']
range_ = [500, 500, 500, 500]

for i, axe in enumerate(axes):
    for j, ax in enumerate(axe):
        # --- Create Twin Axis for Accuracy ---
        ax_acc = ax.twinx()
        # ax_acc.set_ylim(0, 0.1)

        # --- Plot Target Linearity ---
        for layer_idx in range(n_layers):
            ax.plot(smoothing(train_target_linearity[i * 2 + j][layer_idx][:range_[i * 2 + j]]), 
                    color=colors[layer_idx], 
                    linewidth=4,
                    label=f'Layer {layer_idx+1}')
        # ax.set_ylim(0, 0.6)
            
        # Plot Accuracy as a dashed background reference
        line_train = ax_acc.plot(train_accuracy[i * 2 + j][:range_[i * 2 + j]], color='blue', linestyle='--', alpha=0.4, label='Train Acc', linewidth=4.0)
        line_test = ax_acc.plot(test_accuracy[i * 2 + j][:range_[i * 2 + j]], color='red', linestyle='--', alpha=0.4, label='Test Acc', linewidth=4.0)

        # Formatting
        ax.set_title(titles[i * 2 + j], weight='bold', size=14)
        ax.grid(True, which='both', linestyle=':', alpha=0.5)
        # ax.tick_params(labelleft=True)
        
        if i == 0 and j == len(axe) - 1:
            # Create a combined legend
            lines, labels = ax.get_legend_handles_labels()
            lines2, labels2 = ax_acc.get_legend_handles_labels()
            ax.legend(lines + lines2, labels + labels2, loc='lower right', fontsize='small')


fig.supylabel('Target Linearity', weight='bold', size=18, x=0.02)
fig.text(0.98, 0.4, "Accuracy", weight='bold', size=18, 
         va='center', rotation='vertical', rotation_mode='anchor')
fig.suptitle('Grokking', weight='bold', size=22)
fig.supxlabel('Epochs', weight='bold', size=18)
#plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(ncols=2, nrows=2, figsize=(6.5, 6.5), sharey=True)

titles = ['$\\lambda=0.0$', '$\\lambda=0.25$', '$\\lambda=0.5$', '$\\lambda=0.75$']
range_ = [500, 500, 500, 500]

for i, axe in enumerate(axes):
    for j, ax in enumerate(axe):
        # --- Create Twin Axis for Accuracy ---
        ax_acc = ax.twinx()    
        # ax_acc.set_ylim(-0.05, 1.0) # Keep accuracy bounded
        
        # Plot Accuracy as a dashed background reference
        line_test = ax_acc.plot(generalization_error[i * 2 + j][:range_[i * 2 + j]], color='red', alpha=0.5, label='Generalization Error', linewidth=4)
                
        # --- Plot Target Linearity Gap ---
        ax.plot(smoothing(tl_gap[i * 2 + j][:range_[i * 2 + j]]), 
                color='blue', 
                linewidth=4,
                alpha=0.5,
                label=f'$L3-L2$')

        # ax.set_ylim(ymin=0.0, ymax=0.32)

        # Formatting
        ax.set_title(titles[i * 2 + j], weight='bold', size=16)
        ax.grid(True, which='both', linestyle=':', alpha=0.5)
        ax.tick_params(labelleft=True)
        
        # Create a combined legend
        if i == 0 and j == len(axe) - 1:
                lines, labels = ax.get_legend_handles_labels()
                lines2, labels2 = ax_acc.get_legend_handles_labels()
                ax.legend(lines + lines2, labels + labels2, loc='lower right', fontsize='small')

fig.supylabel('TL Gap', weight='bold', size=18, x=0.02)
fig.text(1.02, 0.25, "Generalization Error", weight='bold', size=18, 
         va='center', rotation='vertical', rotation_mode='anchor')
fig.supxlabel('Epochs', weight='bold', size=18)
plt.tight_layout()
plt.show()